# Tree of Thoughts: Deliberate Problem Solving with Language Models

## Learning Objectives
1. Understand tree-based search for problem-solving
2. Implement beam search and scoring functions
3. Compare greedy (CoT) vs. tree-based (ToT) approaches
4. Design heuristics for pruning bad branches
5. Analyze trade-offs: success rate vs. computation cost

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, deque
from typing import List, Tuple, Dict, Set
import heapq

np.random.seed(42)

print("Tree of Thoughts Implementation")
print("=" * 50)

## Level 1: Greedy vs. Tree Search

Demonstrating why greedy approach fails and tree search succeeds.

In [ ]:
# Simulate a simple puzzle: reach target 24 from [3, 8, 3, 8]

class PuzzleState:
    def __init__(self, numbers: List[float], operations: List[str] = None):
        self.numbers = sorted(numbers)
        self.operations = operations or []
        self.depth = len(self.operations)
    
    def is_goal(self, target: float = 24.0) -> bool:
        """Check if any number equals target."""
        return any(abs(n - target) < 1e-6 for n in self.numbers)
    
    def __repr__(self):
        return f"State({self.numbers}, depth={self.depth})"

def generate_candidates(state: PuzzleState, max_candidates: int = 5) -> List[Tuple[PuzzleState, str, float]]:
    """Generate possible next states with scores."""
    candidates = []
    target = 24.0
    
    for i in range(len(state.numbers)):
        for j in range(len(state.numbers)):
            if i == j:
                continue
            
            a, b = state.numbers[i], state.numbers[j]
            remaining = [state.numbers[k] for k in range(len(state.numbers)) if k != i and k != j]
            
            # Try operations
            ops = [
                (a + b, f"{a:.1f} + {b:.1f} = {a+b:.1f}"),
                (a - b, f"{a:.1f} - {b:.1f} = {a-b:.1f}"),
                (a * b, f"{a:.1f} * {b:.1f} = {a*b:.1f}")
            ]
            if abs(b) > 1e-6:
                ops.append((a / b, f"{a:.1f} / {b:.1f} = {a/b:.2f}"))
            
            for result, op_str in ops:
                new_nums = remaining + [result]
                new_state = PuzzleState(new_nums, state.operations + [op_str])
                
                # Score: how close to 24?
                closest_to_target = min(abs(n - target) for n in new_nums)
                score = 1.0 / (1.0 + closest_to_target)  # Higher is better
                
                candidates.append((new_state, op_str, score))
    
    return candidates

# Greedy search (CoT-like)
def greedy_search(initial_nums: List[float], max_depth: int = 5) -> Tuple[bool, List[str]]:
    state = PuzzleState(initial_nums)
    
    for depth in range(max_depth):
        if state.is_goal():
            return True, state.operations
        
        if len(state.numbers) == 1:
            return False, state.operations
        
        # Greedy: pick only best candidate
        candidates = generate_candidates(state)
        if not candidates:
            return False, state.operations
        
        best = max(candidates, key=lambda x: x[2])
        state = best[0]
    
    return False, state.operations

# Test
greedy_success, greedy_ops = greedy_search([3, 8, 3, 8])
print("Greedy Search (CoT-like):")
print(f"  Success: {greedy_success}")
print(f"  Operations: {greedy_ops}")
print(f"  Final value: {greedy_ops[-1] if greedy_ops else 'N/A'}")
print()

## Level 2: Beam Search with Tree Exploration

Full tree search with beam width to explore multiple promising branches.

In [ ]:
def beam_search(initial_nums: List[float], beam_width: int = 3, max_depth: int = 5) -> Tuple[bool, List[str]]:
    """Beam search: explore top-K candidates at each level."""
    initial_state = PuzzleState(initial_nums)
    
    # (state, operations)
    current_beam = [(initial_state, initial_state.operations)]
    all_states_explored = 0
    
    for depth in range(max_depth):
        next_beam = []
        
        for state, ops in current_beam:
            # Goal check
            if state.is_goal():
                return True, ops, all_states_explored
            
            # Terminal check
            if len(state.numbers) == 1:
                continue
            
            # Expand
            candidates = generate_candidates(state)
            all_states_explored += len(candidates)
            
            for new_state, op_str, score in candidates:
                next_beam.append((new_state, new_state.operations, score))
        
        if not next_beam:
            break
        
        # Keep top-K by score
        next_beam = sorted(next_beam, key=lambda x: x[2], reverse=True)[:beam_width]
        current_beam = [(state, ops) for state, ops, _ in next_beam]
    
    # Return best found
    if current_beam:
        return False, current_beam[0][1], all_states_explored
    return False, [], all_states_explored

# Test different beam widths
results = {}
for beam_width in [1, 2, 3, 5]:
    success, ops, explored = beam_search([3, 8, 3, 8], beam_width=beam_width)
    results[beam_width] = {"success": success, "ops": ops, "explored": explored}
    print(f"Beam Width {beam_width}:")
    print(f"  Success: {success}")
    print(f"  Operations: {len(ops)}")
    print(f"  States Explored: {explored}")
    print()

## Real-World Example 1: 24-Game with Different Strategies

Solving the 24-game with greedy, beam search, and exhaustive search.

In [ ]:
# Test suite of 24-game problems
test_problems = [
    [3, 8, 3, 8],
    [1, 2, 3, 4],
    [6, 6, 6, 6],
    [8, 8, 8, 8],
    [1, 1, 1, 1]
]

strategy_performance = defaultdict(lambda: {"success": 0, "steps": [], "explored": []})

for problem in test_problems:
    # Greedy
    greedy_success, greedy_ops = greedy_search(problem, max_depth=6)
    strategy_performance["Greedy"]["success"] += int(greedy_success)
    strategy_performance["Greedy"]["steps"].append(len(greedy_ops))
    strategy_performance["Greedy"]["explored"].append(1)  # Greedy explores 1 path per level
    
    # Beam width 3
    beam3_success, beam3_ops, beam3_explored = beam_search(problem, beam_width=3, max_depth=6)
    strategy_performance["Beam (B=3)"]["success"] += int(beam3_success)
    strategy_performance["Beam (B=3)"]["steps"].append(len(beam3_ops))
    strategy_performance["Beam (B=3)"]["explored"].append(beam3_explored)
    
    # Beam width 5
    beam5_success, beam5_ops, beam5_explored = beam_search(problem, beam_width=5, max_depth=6)
    strategy_performance["Beam (B=5)"]["success"] += int(beam5_success)
    strategy_performance["Beam (B=5)"]["steps"].append(len(beam5_ops))
    strategy_performance["Beam (B=5)"]["explored"].append(beam5_explored)

print("Strategy Comparison on 24-Game:")
print("=" * 60)
for strategy, perf in sorted(strategy_performance.items()):
    avg_steps = np.mean(perf["steps"]) if perf["steps"] else 0
    avg_explored = np.mean(perf["explored"]) if perf["explored"] else 0
    print(f"{strategy:15} | Success: {perf['success']}/{len(test_problems)} | Avg Steps: {avg_steps:.1f} | Avg Explored: {avg_explored:.0f}")

## Real-World Example 2: Adaptive Pruning

Dynamically adjust pruning threshold based on problem difficulty.

In [ ]:
def beam_search_with_threshold(initial_nums: List[float], beam_width: int = 3, 
                               min_score: float = 0.3, max_depth: int = 5) -> Tuple[bool, List[str], int]:
    """Beam search with pruning threshold."""
    initial_state = PuzzleState(initial_nums)
    current_beam = [(initial_state, initial_state.operations, 1.0)]
    nodes_explored = 0
    
    for depth in range(max_depth):
        next_beam = []
        
        for state, ops, parent_score in current_beam:
            if state.is_goal():
                return True, ops, nodes_explored
            
            if len(state.numbers) == 1:
                continue
            
            candidates = generate_candidates(state)
            nodes_explored += len(candidates)
            
            # Prune by threshold
            for new_state, op_str, score in candidates:
                if score < min_score:
                    continue  # Prune
                next_beam.append((new_state, new_state.operations, score))
        
        if not next_beam:
            break
        
        # Keep top-K
        next_beam = sorted(next_beam, key=lambda x: x[2], reverse=True)[:beam_width]
        current_beam = [(state, ops, score) for state, ops, score in next_beam]
    
    if current_beam:
        return False, current_beam[0][1], nodes_explored
    return False, [], nodes_explored

# Test different thresholds
threshold_results = {}
for threshold in [0.1, 0.3, 0.5, 0.7]:
    success, ops, explored = beam_search_with_threshold([3, 8, 3, 8], beam_width=3, 
                                                         min_score=threshold, max_depth=6)
    threshold_results[threshold] = {"success": success, "explored": explored}
    print(f"Threshold {threshold:.1f}: Success={success}, Explored={explored} nodes")

## Real-World Example 3: Analyzing Search Trees

Visualizing and analyzing the search tree to understand pruning and exploration.

In [ ]:
class TreeAnalyzer:
    """Analyze search trees and generate statistics."""
    
    def __init__(self, initial_nums: List[float]):
        self.initial = PuzzleState(initial_nums)
        self.nodes_by_depth = defaultdict(int)
        self.solutions_by_depth = defaultdict(int)
    
    def analyze_tree(self, beam_width: int, max_depth: int) -> Dict:
        """Analyze tree structure."""
        current_beam = [self.initial]
        self.nodes_by_depth = defaultdict(int)
        self.solutions_by_depth = defaultdict(int)
        
        for depth in range(max_depth):
            self.nodes_by_depth[depth] = len(current_beam)
            
            # Count solutions at this depth
            solution_count = sum(1 for state in current_beam if state.is_goal())
            self.solutions_by_depth[depth] = solution_count
            
            next_beam = []
            for state in current_beam:
                if state.is_goal() or len(state.numbers) == 1:
                    continue
                
                candidates = generate_candidates(state)
                scored = [(c[0], c[2]) for c in candidates]
                top_k = sorted(scored, key=lambda x: x[1], reverse=True)[:beam_width]
                next_beam.extend([state for state, _ in top_k])
            
            if not next_beam:
                break
            current_beam = next_beam
        
        return {
            "nodes_by_depth": dict(self.nodes_by_depth),
            "solutions_by_depth": dict(self.solutions_by_depth)
        }

analyzer = TreeAnalyzer([3, 8, 3, 8])
result = analyzer.analyze_tree(beam_width=3, max_depth=5)

print("Tree Analysis:")
print("=" * 60)
for depth in sorted(result["nodes_by_depth"].keys()):
    nodes = result["nodes_by_depth"][depth]
    solutions = result["solutions_by_depth"].get(depth, 0)
    print(f"Depth {depth}: {nodes} nodes explored, {solutions} solutions found")

## Comparison and Visualization

Visualizing the trade-off between accuracy and computation cost.

In [ ]:
# Simulate performance across different budgets
simulation_data = {
    "Greedy (B=1)": {"accuracy": 0.15, "nodes": 10},
    "Light (B=2, D=3)": {"accuracy": 0.35, "nodes": 40},
    "Medium (B=3, D=4)": {"accuracy": 0.60, "nodes": 120},
    "Heavy (B=5, D=5)": {"accuracy": 0.85, "nodes": 1500},
    "Exhaustive (All)": {"accuracy": 1.0, "nodes": 5000}
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy vs Computation
strategies = list(simulation_data.keys())
accuracy = [simulation_data[s]["accuracy"] * 100 for s in strategies]
nodes = [simulation_data[s]["nodes"] for s in strategies]

colors = ['steelblue', 'orange', 'green', 'red', 'purple']
for i, strategy in enumerate(strategies):
    axes[0].scatter(nodes[i], accuracy[i], s=300, alpha=0.7, color=colors[i], edgecolors='black', linewidth=2)
    axes[0].annotate(strategy, (nodes[i], accuracy[i]), xytext=(5, 5), textcoords='offset points', fontsize=9)

axes[0].set_xlabel('Nodes Explored', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Success Rate (%)', fontsize=12, fontweight='bold')
axes[0].set_title('Accuracy vs. Computation Trade-off', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 105])
axes[0].set_xscale('log')

# Plot 2: Efficiency (accuracy per node)
efficiency = [acc / np.log(nodes[i] + 1) for i, acc in enumerate(accuracy)]
axes[1].bar(range(len(strategies)), efficiency, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1].set_xticks(range(len(strategies)))
axes[1].set_xticklabels(strategies, rotation=45, ha='right', fontsize=9)
axes[1].set_ylabel('Efficiency (Accuracy / log(Nodes))', fontsize=12, fontweight='bold')
axes[1].set_title('Algorithm Efficiency', fontsize=13, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\\nKey Observations:")
print("=" * 60)
print("Greedy (B=1):   Fast but fails on hard problems (15% success)")
print("Light (B=2):    Quick iteration, moderate success (35%)")
print("Medium (B=3):   Good balance of speed and accuracy (60%)")
print("Heavy (B=5):    High accuracy but expensive (85%)")
print("Exhaustive:     100% success but prohibitively expensive (5000+ nodes)")
print("\\nRecommendation: Use B=3-5 for most problems. Start with B=3, increase if needed.")

## Key Takeaways

**Tree of Thoughts enables structured problem-solving through search.**

### Core Insights
1. **Greedy fails:** Single path doesn't work for complex problems
2. **Tree succeeds:** Exploring multiple options + pruning enables solutions
3. **Beam search:** Balance between exhaustive and greedy
4. **Scoring matters:** Good heuristics enable effective pruning

### When to Use Tree of Thoughts
- Hard problems where CoT fails (success <50%)
- Problems with multiple solution paths
- Tasks requiring planning and backtracking
- When you have good scoring heuristics

### Budget Allocation
- **Beam Width (B):** 3-5 typical; increase if exploring too greedily
- **Depth (D):** 3-5 typical; increase if solutions are deep
- **Total Cost:** ~B^D nodes explored

### Production Lessons
- Only use ToT for hard problems (CoT insufficient)
- Cache scored states (avoid re-evaluation)
- Monitor exploration budget (don't exceed max nodes)
- Combine with domain-specific heuristics (faster pruning)

### Related Concepts
- [Chain-of-Thought](./01-chain-of-thought.md) — Linear reasoning (base case)
- [ReAct](./02-react.md) — Reasoning with tool use (can be combined with ToT)